# 01 Data Exploration
## Stock Data Analysis and Visualization

This notebook explores historical stock data, performs statistical analysis, and visualizes trends.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.data_loader import download_stock_data, get_data_stats, load_stock_data_from_csv
from src.preprocessing import DataPreprocessor

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
print("Libraries imported successfully!")

## Step 1: Download Stock Data

In [ ]:
# Configure stock data parameters
ticker = 'AAPL'
start_date = '2020-01-01'
end_date = '2026-02-01'

# Download data
data = download_stock_data(ticker, start_date, end_date)
print(f"\nData shape: {data.shape}")
print(f"\nFirst few rows:")
print(data.head())

## Step 2: Basic Statistics

In [ ]:
# Get detailed statistics
get_data_stats(data, ticker)

# Display numerical statistics
print("\nNumerical Summary:")
print(data.describe())

## Step 3: Price Trends Visualization

In [ ]:
# Plot closing price
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Closing price
axes[0].plot(data.index, data['Close'], linewidth=2, color='blue', label='Close Price')
axes[0].fill_between(data.index, data['Close'], alpha=0.3)
axes[0].set_title(f'{ticker} Closing Price Over Time', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price ($)', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# OHLC price ranges
axes[1].fill_between(data.index, data['Low'], data['High'], alpha=0.3, label='High-Low Range')
axes[1].plot(data.index, data['Close'], linewidth=2, color='red', label='Close')
axes[1].set_title(f'{ticker} High-Low Range', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Price ($)', fontsize=12)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Price Range: ${data['Close'].min():.2f} - ${data['Close'].max():.2f}")

## Step 4: Volume Analysis

In [ ]:
# Volume analysis
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Volume bars
axes[0].bar(data.index, data['Volume'], color='steelblue', alpha=0.6, label='Volume')
axes[0].set_title(f'{ticker} Trading Volume', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Volume', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].legend()

# Volume vs Price
color = 'tab:blue'
axes[1].bar(data.index, data['Volume'], color='steelblue', alpha=0.4)
ax2 = axes[1].twinx()
color = 'tab:red'
ax2.plot(data.index, data['Close'], color=color, linewidth=2, label='Close Price')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Volume', fontsize=12, color='steelblue')
ax2.set_ylabel('Price ($)', fontsize=12, color=color)
axes[1].set_title(f'{ticker} Volume vs Price', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Average Daily Volume: {data['Volume'].mean():,.0f}")
print(f"Median Daily Volume: {data['Volume'].median():,.0f}")

## Step 5: Returns Distribution

In [ ]:
# Calculate daily returns
data['Daily_Return'] = data['Close'].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Returns histogram
axes[0].hist(data['Daily_Return'].dropna(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_title('Distribution of Daily Returns', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Daily Return (%)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].axvline(data['Daily_Return'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {data["Daily_Return"].mean():.2f}%')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Returns over time
axes[1].plot(data.index[1:], data['Daily_Return'][1:], linewidth=1, alpha=0.7, color='steelblue')
axes[1].axhline(0, color='black', linestyle='-', linewidth=0.8)
axes[1].fill_between(data.index[1:], data['Daily_Return'][1:], 0, 
                      where=(data['Daily_Return'][1:] >= 0), alpha=0.3, color='green', label='Positive')
axes[1].fill_between(data.index[1:], data['Daily_Return'][1:], 0, 
                      where=(data['Daily_Return'][1:] < 0), alpha=0.3, color='red', label='Negative')
axes[1].set_title('Daily Returns Over Time', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Return (%)', fontsize=12)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nReturn Statistics:")
print(f"Mean Daily Return: {data['Daily_Return'].mean():.4f}%")
print(f"Std Dev: {data['Daily_Return'].std():.4f}%")
print(f"Min Return: {data['Daily_Return'].min():.2f}%")
print(f"Max Return: {data['Daily_Return'].max():.2f}%")

## Step 6: Technical Indicators

In [ ]:
# Add technical indicators
preprocessor = DataPreprocessor()
data_with_indicators = preprocessor.add_technical_indicators(data.copy())

# Visualize Moving Averages
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(data_with_indicators.index, data_with_indicators['Close'], 
        label='Close Price', linewidth=2, color='black', alpha=0.7)
ax.plot(data_with_indicators.index, data_with_indicators['SMA_20'], 
        label='SMA 20', linewidth=2, alpha=0.7)
ax.plot(data_with_indicators.index, data_with_indicators['SMA_50'], 
        label='SMA 50', linewidth=2, alpha=0.7)
ax.plot(data_with_indicators.index, data_with_indicators['EMA_20'], 
        label='EMA 20', linewidth=2, alpha=0.7, linestyle='--')

ax.set_title(f'{ticker} - Moving Averages', fontsize=14, fontweight='bold')
ax.set_ylabel('Price ($)', fontsize=12)
ax.set_xlabel('Date', fontsize=12)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 7: RSI Indicator

In [ ]:
# RSI Analysis
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Price
axes[0].plot(data_with_indicators.index, data_with_indicators['Close'], 
             label='Close Price', linewidth=2, color='steelblue')
axes[0].set_title(f'{ticker} - Price and RSI', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price ($)', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# RSI
axes[1].plot(data_with_indicators.index, data_with_indicators['RSI'], 
             label='RSI(14)', linewidth=2, color='orange')
axes[1].axhline(70, color='red', linestyle='--', linewidth=1, label='Overbought (70)')
axes[1].axhline(30, color='green', linestyle='--', linewidth=1, label='Oversold (30)')
axes[1].fill_between(data_with_indicators.index, 30, 70, alpha=0.1, color='gray')
axes[1].set_ylabel('RSI', fontsize=12)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylim(0, 100)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"RSI Statistics:")
print(f"Current RSI: {data_with_indicators['RSI'].iloc[-1]:.2f}")
print(f"Average RSI: {data_with_indicators['RSI'].mean():.2f}")

## Step 8: Correlation Analysis

In [ ]:
# Correlation heatmap
numeric_cols = data_with_indicators.select_dtypes(include=[np.number]).columns
correlation_matrix = data_with_indicators[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, cbar_kws={'label': 'Correlation'})
plt.title(f'{ticker} - Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

Key findings from data exploration:
- Stock data successfully loaded
- Technical indicators calculated
- Trends and patterns identified
- Ready for model training!